<a href="https://colab.research.google.com/github/pathilink/adyen_payment_optimization_case/blob/main/notebooks/02_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <font color='#0ABF56'>Optimisation Data Analyst Case Study </font>

## <font color='#0ABF56'> 02 - Data Cleaning </font>

# Libraries

In [1]:
import pandas as pd
# import numpy as np
# import datetime
# import seaborn as sns
# from matplotlib import pyplot as plt

# Data

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
df = pd.read_csv('/content/drive/MyDrive/test/adyen/data/raw/DataAnalyst_case_study_data.csv')

# Cleaning

In [4]:
def clean_transactions(df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns a new cleaned dataframe.
    """

    # create copy to avoid modifying original dataframe
    df_clean = df.copy()

    # drop constant / unnecessary columns
    df_clean.drop(
        columns=['currency_code', 'received'],
        inplace=True,
        errors='ignore'
    )

    # int -> string
    df_clean['psp_reference'] = df_clean['psp_reference'].astype(str)

    # int -> string
    df_clean['bin'] = (
        df_clean['bin']
        .astype('Int64')
        .astype(str)
    )

    # object -> float
    df_clean['amount'] = (
        df_clean['amount']
        .astype(str)
        .str.replace(',', '', regex=False)
        .astype(float)
    )

    # Yes / No -> Boolean
    mapping_bool = {'Yes': True, 'No': False}

    bool_columns = [
        'avs_data_supplied',
        'cvc_data_supplied'
    ]

    for col in bool_columns:
        df_clean[col] = df_clean[col].map(mapping_bool)

    # 1 / 0 -> Boolean
    mapping_auth = {
        1: True,
        0: False
    }

    df_clean['authorization'] = (
        df_clean['authorization']
        .map(mapping_auth)
    )

    # object -> datetime
    df_clean['creation_date'] = pd.to_datetime(
        df_clean['creation_date'],
        format='%m/%d/%y %H:%M'
    )

    return df_clean

In [6]:
df_clean = clean_transactions(df)
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 933387 entries, 0 to 933386
Data columns (total 11 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   psp_reference          933387 non-null  object        
 1   bin                    933387 non-null  object        
 2   scheme                 933387 non-null  object        
 3   issuername             932580 non-null  object        
 4   shopper_interaction    933387 non-null  object        
 5   avs_data_supplied      933387 non-null  bool          
 6   cvc_data_supplied      933387 non-null  bool          
 7   amount                 933387 non-null  float64       
 8   raw_acquirer_response  933387 non-null  object        
 9   creation_date          933387 non-null  datetime64[ns]
 10  authorization          933387 non-null  bool          
dtypes: bool(3), datetime64[ns](1), float64(1), object(6)
memory usage: 59.6+ MB


In [7]:
df_clean.head()

,psp_reference,bin,scheme,issuername,shopper_interaction,avs_data_supplied,cvc_data_supplied,amount,raw_acquirer_response,creation_date,authorization
0,1,400178,visa,BANCO DO BRASIL S.A.,Ecommerce,False,False,1.00,05 : Do not honor / A201 : 3D Secure Mandated,2019-06-01 00:19:00,False
1,2,486348,visa,FIRST ATLANTIC BANK LIMITED,Ecommerce,False,False,6.48,00 : Approved or completed successfully,2019-06-01 00:22:00,True
2,3,482481,visa,ITAU UNIBANCO S.A.,Ecommerce,False,True,4.00,06 : Error,2019-06-01 00:46:00,False
3,4,439267,visa,CAIXA ECONOMICA FEDERAL,Ecommerce,False,True,2.76,05 : Do not honor / A201 : 3D Secure Mandated,2019-06-01 01:02:00,False
4,5,489347,visa,VTB BANK PJSC,Ecommerce,False,True,97.00,00 : Approved or completed successfully,2019-06-01 01:30:00,True


# Download

In [8]:
output_path = '/content/drive/MyDrive/test/adyen/data/processed/adyen_transactions_clean.csv'

df_clean.to_csv(output_path, index=False)